# Credit Monitoring Agents

Run these cells in order with the project's Python 3.14 kernel. This notebook is
independent of `02-rag.ipynb`; no variables or execution from that notebook are needed.

We run `CreditAssessmentAgent` in two configurations:

1. **Document research:** discover filings, read extracted facts, and retrieve cited passages.
2. **Borrower assessment:** read financials, calculate covenant compliance and headroom,
   and request a prediction from the configured model.

The OpenAI Agents SDK manages the loop: the model selects a tool, receives its result,
then selects another tool or returns an answer. These are two configurations of the
same agent, rather than a multi-agent handoff.

Before starting, run `uv sync --group dev` from the repository root and select the
project's `.venv` kernel. Set `OPENAI_API_KEY` in the environment or root `.env` file.
Set `SEC_USER_AGENT` to your name/organization and contact email if SEC downloads are
needed. Agent questions call the OpenAI API; document processing may also call the
extraction model. Existing local documents and successful extractions are reused.

## 1. Set up paths and configuration

Start Jupyter from the repository root or `notebooks/`. Paths below resolve to the
same data directory in either case. Model and database choices stay in application
configuration, outside model-supplied tool arguments.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from credit_monitoring.observability.logging import configure_logging

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "data/public_sec_documents.csv").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks/.")

load_dotenv(root / ".env")
configure_logging()

MODEL = "gpt-4o-mini"
MAX_TOOL_ROUNDS = 6
TICKER = "FMC"
BORROWER_ID = "SYN002"
PERIOD_END = "2025-09-30"
INFORMATION_CUTOFF = "2025-09-30"

# The real SEC issuer and synthetic borrower are separate demonstrations.
data_dir = root / "data"
source_file = data_dir / "public_sec_documents.csv"
html_dir = data_dir / "sec_documents/html"
markdown_dir = data_dir / "sec_documents/markdown"
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
sec_user_agent = os.environ.get("SEC_USER_AGENT", "")

print("Repository:", root)
print("Agent model:", MODEL)

## 2. Create the client and document service

Creating the service initializes its local SQLite storage. Listing catalog documents
reads local state without downloading filings or calling a model. Document tools
can process an individual filing later if the question needs missing or stale facts;
there is no prerequisite to process the full catalog.

In [ ]:
from openai import OpenAI

from credit_monitoring.application.document_processing_service import DocumentProcessingService

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the environment or the root .env file, then rerun.")

openai_client = OpenAI()
processing_service = DocumentProcessingService(
    client=openai_client,
    model=MODEL,
    source_file=source_file,
    html_dir=html_dir,
    markdown_dir=markdown_dir,
    database_path=processed_dir / "covenants.sqlite3",
    user_agent=sec_user_agent,
)

In [ ]:
document_states = processing_service.list_document_extractions(
    ticker=TICKER,
    include_results=False,
)
display(pd.DataFrame([
    {
        "document_id": document.document_id,
        "ticker": document.ticker,
        "status": document.status,
        "error": document.error,
    }
    for document in document_states
]))

## 3. Inspect the document tools

The reusable `DocumentTools` class owns its service and issuer scope. Its `.tools`
property wraps bound methods with `function_tool()`. Method signatures define the
arguments and docstrings describe when and how to use each tool. The SDK supplies
`ToolContext`; it is not a model-visible argument.

In [ ]:
from credit_monitoring.agents.tools.documents import DocumentTools


def show_tools(tools):
    display(pd.DataFrame([
        {
            "tool": tool.name,
            "description": tool.description.splitlines()[0],
            "arguments": ", ".join(tool.params_json_schema["properties"]),
        }
        for tool in tools
    ]))


show_tools(DocumentTools(processing_service, ticker=TICKER).tools)

## 4. Run the document research agent

Use `await agent.ask_async(...)` in notebooks because Jupyter already runs an event
loop. This question asks the agent to discover documents, read complete schedules,
and find supporting passages. Actual tool selection is model-driven and appears in
the trace below. The agent may process a filing if its extraction is unavailable.

Document extraction is currently unverified: `validation: null` means automatic
post-extraction verification did not run. The answer should retain gaps and conflicting
versions rather than deciding legal precedence or calculating ratios from prose.

In [ ]:
from credit_monitoring.agents.credit_assessment import CreditAssessmentAgent

# Four document tools; no assessment service is required for this configuration.
document_agent = CreditAssessmentAgent(
    client=openai_client,
    service=processing_service,
    model=MODEL,
    max_tool_rounds=MAX_TOOL_ROUNDS,
)

In [ ]:
document_question = (
    f"List the available {TICKER} documents and read the complete leverage and "
    "interest-coverage schedules in FMC Amendment No. 3. Include every explicitly "
    "named fiscal quarter, then search the filings for supporting passages and "
    "explicitly reported ratios. Cite the sources. Do not determine which "
    "amendment governs or calculate headroom."
)
document_answer = await document_agent.ask_async(document_question, ticker=TICKER)

## 5. Inspect answers, sources, and the actual tool trace

`AgentAnswer` contains the answer, completion/error status, returned source citations,
and executed tool outcomes. The trace records actual calls, including errors and cache
hits; it excludes private model reasoning. An incomplete result may still contain useful
sources and traces.

In [ ]:
def show_answer(outcome):
    display(Markdown(outcome.answer))
    print("Completed:", outcome.complete, "Error:", outcome.error)
    if outcome.sources:
        display(pd.DataFrame([source.model_dump() for source in outcome.sources]))
    display(pd.DataFrame([
        {
            "step": step,
            "tool": execution.name,
            "arguments": execution.arguments,
            "ok": execution.outcome.get("ok"),
            "error": execution.outcome.get("error"),
        }
        for step, execution in enumerate(outcome.tool_trace, start=1)
    ]))


show_answer(document_answer)

# For the complete JSON payload, including extraction status and cache_hit:
# print(document_answer.model_dump_json(indent=2))

## 6. Load synthetic financials and configure assessments

This example uses `SYN002` from the existing synthetic input CSVs. It has no inferred
relationship to FMC. We pass only `borrower_id` when assessing it.

Financial loading uses `agent_financials.sqlite3`; calculated results and predictions
use the separate `agent_analytics.sqlite3` database. Reloading the CSV upserts the demo
financial snapshots. Each assessment tool call appends a new assessment run.

The default model is a stub: it returns unknown risk and no probability. Deterministic
covenant calculations still run. Structured assessment verification is active.

In [ ]:
from credit_monitoring.application.assessment_service import AssessmentService
from credit_monitoring.config.settings import AssessmentSettings, RiskModelConfig
from credit_monitoring.covenants.structured import SyntheticCovenantReader
from credit_monitoring.financials.repositories import FinancialRepository
from credit_monitoring.ingestion.loaders.financials import load_financials

agent_financial_db = processed_dir / "agent_financials.sqlite3"
load_result = load_financials(
    data_dir / "synthetic_quarterly_financials.csv",
    agent_financial_db,
)
print(load_result)

risk_model = RiskModelConfig(type="stub")
# Optional, after training the baseline: replace the line above with
# risk_model = RiskModelConfig(
#     type="logistic_regression",
#     version="v1",
#     artifact_path=data_dir / "models/logistic_regression/v1",
# )

assessment_service = AssessmentService(
    financial_repository=FinancialRepository(agent_financial_db),
    covenant_reader=SyntheticCovenantReader(data_dir / "synthetic_covenant_terms.csv"),
    settings=AssessmentSettings(
        analytics_database=processed_dir / "agent_analytics.sqlite3",
        risk_model=risk_model,
    ),
)

## 7. Create the borrower assessment agent

Supplying the assessment service configures the agent for calculations. The three
assessment tools are enabled on questions that also supply an explicit `borrower_id`.
They accept only reporting date and information cutoff; the application owns borrower
scope, formulas, database paths, and model selection.

In [ ]:
from credit_monitoring.agents.tools.assessment import AssessmentTools

assessment_agent = CreditAssessmentAgent(
    client=openai_client,
    service=processing_service,
    assessment_service=assessment_service,
    model=MODEL,
    max_tool_rounds=MAX_TOOL_ROUNDS,
)

# These are the seven tools available to a borrower-scoped question.
show_tools(
    DocumentTools(processing_service).tools
    + AssessmentTools(assessment_service, borrower_id=BORROWER_ID).tools
)

## 8. Run calculations and request a prediction

The question supplies explicit dates. `get_financials` reads inputs, `assess_covenants`
performs calculations without ML, and `predict_risk` performs calculations with the
configured model. The model chooses which calls to make; inspect the trace to see what
ran. Current compliance and next-quarter predictions must be reported separately.

In [ ]:
assessment_question = (
    f"For borrower {BORROWER_ID}, read the financials and assess covenant compliance "
    f"and headroom for {PERIOD_END}, with an information cutoff of {INFORMATION_CUTOFF}. "
    "Also request a next-quarter covenant breach prediction. Cite financial and "
    "covenant provenance, distinguish current compliance from predictions, and "
    "explain if the configured model has no forecast."
)
assessment_answer = await assessment_agent.ask_async(
    assessment_question,
    borrower_id=BORROWER_ID,
)
show_answer(assessment_answer)

In [ ]:
# Inspect calculation/prediction metadata without interpreting the prose answer.
for execution in assessment_answer.tool_trace:
    if execution.name not in {"assess_covenants", "predict_risk"}:
        continue
    if not execution.outcome.get("ok"):
        print(execution.name, execution.outcome)
        continue
    assessment_data = execution.outcome["data"]
    print("Tool:", execution.name)
    print("Assessment run:", assessment_data["assessment_run_id"])
    print("ML status:", assessment_data["ml_status"])
    print("Verification:", assessment_data["verification"])
    print("Prediction:", assessment_data["prediction"])

# Optional: inspect or save the structured outcome explicitly.
# print(assessment_answer.model_dump_json(indent=2))

## 9. Optional trained-model prediction and further questions

To obtain a probability from the synthetic Logistic Regression baseline, run these
commands from the repository root:

```bash
uv sync --extra ml --group dev
uv run --extra ml python scripts/generate_training_dataset.py
uv run --extra ml python scripts/train_risk_model.py
```

Select the Logistic Regression configuration in step 6, then rerun steps 6–8 to rebuild
the service and agent. This baseline is trained on simulated histories; its probabilities
are not validated on real credit outcomes. A missing or incompatible artifact returns an
explicit prediction failure while preserving valid calculations. See
[the ML workflow](../docs/ml_early_warning.md) for the complete simulated-borrower example.

To ask another question, change `document_question` or `assessment_question` and rerun
its agent cell. Each `ask_async()` starts a fresh conversation; include all required
scope and dates in each question. The default allows six tool rounds followed by one
final model turn. Increase `MAX_TOOL_ROUNDS` and recreate the agent if an intentionally
longer workflow is needed.

For the course submission, show the agent construction, available tools, one completed
answer, and its actual tool trace. [The RAG notebook](02-rag.ipynb) retains the original
`search → build_prompt → llm → rag` workflow for comparison.